# Proyecto: Recomendación de plan Megaline (Smart vs. Ultra)

## Objetivo general

Megaline quiere migrar a sus clientes de planes heredados hacia sus planes nuevos (Smart o Ultra). El objetivo de este proyecto es construir un modelo de clasificación que, a partir del comportamiento mensual de un cliente (llamadas, minutos, mensajes, datos móviles), recomiende el plan que mejor se ajusta a su uso real. El proyecto se considera exitoso si el modelo alcanza una exactitud igual o mayor a 0.75 sobre datos que nunca vio durante el entrenamiento.

## Etapas del proyecto

1. **Carga y exploración de datos**: revisar la calidad y estructura del dataset.
2. **Segmentación de datos**: dividir el dataset en entrenamiento, validación y prueba.
3. **Investigación de hiperparámetros**: comparar árbol de decisión, bosque aleatorio y regresión logística para elegir el modelo más preciso.
4. **Evaluación con el conjunto de prueba**: medir la exactitud final del modelo ganador sobre datos nunca antes vistos.
5. **Prueba de cordura**: comparar el modelo final contra un KNN simple, para confirmar que realmente aporta valor.
6. **Conclusiones generales**: sintetizar los hallazgos de cada etapa.

## Paso 1: Carga y exploración de datos

**Objetivo:** cargar el dataset de comportamiento de clientes y verificar que esté limpio (sin nulos ni duplicados) y listo para modelar, antes de invertir tiempo entrenando modelos sobre datos de mala calidad.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split  # para dividir el dataset en train/valid/test
from sklearn.tree import DecisionTreeClassifier  # modelo 1: árbol de decisión
from sklearn.ensemble import RandomForestClassifier  # modelo 2: bosque aleatorio
from sklearn.linear_model import LogisticRegression  # modelo 3: regresión logística
from sklearn.neighbors import KNeighborsClassifier  # modelo usado en la prueba de cordura
from sklearn.metrics import accuracy_score  # métrica de exactitud para clasificación


In [ ]:
# cargamos el dataset de comportamiento de clientes
df = pd.read_csv('users_behavior.csv')

In [ ]:
# realizamos análisis exploratorio de los datos
print(df.info())  # tipos de dato y conteo de valores no nulos por columna
print()
print(df.head())  # primeras filas para inspeccionar el formato
print()
print(df.describe())  # estadísticas descriptivas (media, min, max, percentiles)
print()
print(f'Valores nulos: {df.isnull().sum()}')  # verificamos que no falten datos
print()
print(f'Valores duplicados: {df.duplicated().sum()}')  # verificamos que no haya filas repetidas
print()
print(f'Shape: {df.shape}')  # cantidad de filas y columnas del dataset


**Conclusión:** el dataset contiene 3214 registros y 5 columnas, sin valores nulos ni duplicados. Al no requerir limpieza adicional, podemos pasar directamente a segmentar los datos para el modelado.

## Paso 2: Segmentación de datos

**Objetivo:** dividir el dataset en tres conjuntos independientes (entrenamiento, validación y prueba) en proporción 60/20/20, para poder comparar modelos de forma justa y reservar una porción de datos nunca vista para la evaluación final.

In [ ]:
# segmentamos los datos en features (variables de entrada) y target (lo que queremos predecir)
features = df.drop(['is_ultra'], axis=1)
target = df['is_ultra']

# primer split: 60% entrenamiento / 40% resto (que luego partimos en validación y prueba)
features_train, features_rest, target_train, target_rest = train_test_split(
    features, target, test_size=0.4, random_state=12345
)

# segundo split: dividimos el 40% "resto" a la mitad -> 20% validación / 20% prueba
features_valid, features_test, target_valid, target_test = train_test_split(
    features_rest, target_rest, test_size=0.5, random_state=12345
)

print(f'Train: {features_train.shape}, Valid: {features_valid.shape}, Test: {features_test.shape}')


**Conclusión:** quedaron 1928 observaciones para entrenamiento y 643 para validación y para prueba, respectivamente — suficientes en los tres conjuntos para entrenar modelos y evaluarlos con confianza.

## Paso 3: Investigación de hiperparámetros

**Objetivos:** entrenar y comparar tres modelos de clasificación (árbol de decisión, bosque aleatorio y regresión logística), variando sus hiperparámetros principales, y seleccionar el que obtenga mayor exactitud sobre el conjunto de validación.

### Árbol de decisión

In [ ]:
# variables para guardar el mejor resultado encontrado durante la búsqueda
tree_best_score = 0
tree_best_depth = 0

# probamos profundidades de 1 a 10 y nos quedamos con la que mejor exactitud da en validación
for depth in range(1, 11):
    model = DecisionTreeClassifier(max_depth=depth, random_state=12345)
    model.fit(features_train, target_train)  # entrenamos solo con el 60% de train
    predictions = model.predict(features_valid)  # predecimos sobre el 20% de validación
    result = accuracy_score(target_valid, predictions)  # comparamos predicción vs. etiqueta real
    if result > tree_best_score:
        tree_best_score = result
        tree_best_depth = depth

print(f'max_depth={tree_best_depth}, exactitud en validación={tree_best_score}')


### Bosque aleatorio

In [ ]:
# variables para guardar la mejor combinación de hiperparámetros encontrada
forest_best_score = 0
forest_best_depth = 0
forest_best_est = 0

# bucle anidado: probamos cada combinación de profundidad (1-10) y número de árboles (10, 20, 30, 40, 50)
for depth in range(1, 11):
    for est in range(10, 51, 10):
        model = RandomForestClassifier(n_estimators=est, max_depth=depth, random_state=12345)
        model.fit(features_train, target_train)  # entrenamos con el 60% de train
        predictions = model.predict(features_valid)  # predecimos sobre el 20% de validación
        result = accuracy_score(target_valid, predictions)
        if result > forest_best_score:
            forest_best_score = result
            forest_best_depth = depth
            forest_best_est = est

print(f'n_estimators={forest_best_est}, max_depth={forest_best_depth}, exactitud en validación={forest_best_score}')


### Regresión logística

In [ ]:
# usamos liblinear porque las variables no están en la misma escala
model = LogisticRegression(random_state=12345, solver='liblinear')
model.fit(features_train, target_train)
predictions = model.predict(features_valid)
logreg_score = accuracy_score(target_valid, predictions)
print(f'Regresión logística, exactitud en validación={logreg_score}')


In [ ]:
# armamos una tabla comparativa con los 3 modelos, ordenada de mayor a menor exactitud
resultados = pd.DataFrame({
    'modelo': ['Bosque aleatorio', 'Árbol de decisión', 'Regresión logística'],
    'exactitud_validacion': [forest_best_score, tree_best_score, logreg_score]
}).sort_values('exactitud_validacion', ascending=False).reset_index(drop=True)

resultados


**Conclusión:** el bosque aleatorio (`n_estimators=40`, `max_depth=8`) fue el modelo más preciso en validación (0.8087), superando al árbol de decisión (0.7854) y a la regresión logística (0.7574). Es el modelo que llevamos a la evaluación final.

## Paso 4: Evaluación con el conjunto de prueba

**Objetivo:** confirmar que el modelo ganador (bosque aleatorio) generaliza bien a datos completamente nuevos, reentrenándolo con train + validación (80%) y midiendo su exactitud sobre el conjunto de prueba (20%), que no participó en ningún paso anterior.

In [ ]:
# unimos train + validación (80%) para entrenar el modelo final con más datos
features_train_full = pd.concat([features_train, features_valid])
target_train_full = pd.concat([target_train, target_valid])

# reentrenamos el bosque aleatorio con sus mejores hiperparámetros (hallados en el Paso 3)
final_model = RandomForestClassifier(n_estimators=forest_best_est, max_depth=forest_best_depth, random_state=12345)
final_model.fit(features_train_full, target_train_full)

# evaluamos sobre el 20% de prueba, que nunca se usó ni para entrenar ni para elegir hiperparámetros
test_predictions = final_model.predict(features_test)
test_accuracy = accuracy_score(target_test, test_predictions)
print(f'Exactitud en el conjunto de prueba: {test_accuracy}')


**Conclusión:** el modelo final alcanzó una exactitud de 0.7994 sobre el conjunto de prueba, superando el umbral de 0.75 exigido por el proyecto y confirmando que el resultado obtenido en validación se sostiene sobre datos nunca vistos.

## Paso 5: Prueba de cordura (KNN)

**Objetivo:** comparar el modelo final contra un `KNeighborsClassifier` simple, entrenado y evaluado con los mismos datos, para verificar que el bosque aleatorio realmente aporta valor y no solo se beneficia del desbalance de clases.

In [ ]:
# entrenamos KNN con los mismos datos (80%) que el modelo final, para una comparación justa
knn_model = KNeighborsClassifier()
knn_model.fit(features_train_full, target_train_full)
knn_predictions = knn_model.predict(features_test)
knn_accuracy = accuracy_score(target_test, knn_predictions)

# comparamos ambas exactitudes sobre el mismo conjunto de prueba
print(f'KNN (prueba de cordura), exactitud en prueba: {knn_accuracy}')
print(f'Bosque aleatorio (modelo final), exactitud en prueba: {test_accuracy}')


**Conclusión:** KNN alcanzó 0.7527 de exactitud — apenas por encima del umbral de 0.75, y claramente inferior al 0.7994 del bosque aleatorio. Esto confirma que el modelo elegido aporta valor real.

## Conclusiones generales

Tras evaluar tres modelos de clasificación variando sus hiperparámetros, el **bosque aleatorio** (`n_estimators=40`, `max_depth=8`) resultó el más preciso, con 0.8087 de exactitud en validación — por encima del árbol de decisión (0.7854) y de la regresión logística (0.7574).

Reentrenado con el 80% de los datos y evaluado sobre el 20% de prueba nunca antes visto, el modelo final obtuvo una exactitud de **0.7994**, superando cómodamente el umbral de 0.75 exigido por el proyecto.

Como prueba de cordura, se comparó contra un `KNeighborsClassifier` simple, que alcanzó 0.7527 — notablemente inferior al bosque aleatorio. Esto confirma que el modelo elegido aporta valor real y no simplemente aprovecha el desbalance de clases (~70% Smart / 30% Ultra) del dataset.

En conjunto, el modelo cumple el objetivo planteado al inicio del proyecto: recomendar de forma confiable el plan (Smart o Ultra) que mejor se ajusta al comportamiento de un cliente, con una exactitud de aproximadamente 80%.